In [1]:
import numpy as np 
import pandas as pd   
import matplotlib.pyplot as plt   

In [2]:
df = pd.read_csv(
    r'C:\Users\boser\Desktop\Project_Carbon\data\processed\final_dataset.csv',
    index_col='date',
    parse_dates=True
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (589, 9)
Columns: ['carbon_price', 'carbon_price_change_pct', 'gas_price_weekly', 'gas_price_lag1', 'vix', 'co2_ppm', 'co2_change', 'days_to_next_cop', 'is_cop_week']


,carbon_price,carbon_price_change_pct,gas_price_weekly,gas_price_lag1,vix,co2_ppm,co2_change,days_to_next_cop,is_cop_week
date,,,,,,,,,
2015-01-11,7.13,6.259314,3.03,3.09,18.982,400.36,0.077490,323,0
2015-01-18,6.80,-4.628331,3.08,3.03,20.996,399.70,-0.164852,316,0
2015-01-25,7.08,4.117647,2.95,3.08,17.950,400.39,0.172629,309,0
2015-02-01,6.95,-1.836158,2.91,2.95,18.582,400.44,0.012488,302,0
2015-02-08,7.63,9.784173,2.72,2.91,17.846,400.14,-0.074918,295,0


In [3]:
def assign_regime(date):
    if date < pd.Timestamp('2018-01-01'):
        return 0   # Phase 1 — Low price, low volatility, pre-reform
    elif date < pd.Timestamp('2021-01-01'):
        return 1   # Phase 2 — Rising price, policy reform period
    else:
        return 2   # Phase 3 — High price, energy crisis, post-COVID

df['regime'] = [assign_regime(d) for d in df.index]

print("Regime distribution:")
print(df['regime'].value_counts().sort_index())

Regime distribution:
regime
0    156
1    156
2    277
Name: count, dtype: int64


In [5]:
df['recent_volatility'] = df['carbon_price_change_pct'].rolling(window=12).std()

print("Recent volatility nulls:", df['recent_volatility'].isnull().sum())
print("Sample stats:")
print(df['recent_volatility'].describe())

Recent volatility nulls: 11
Sample stats:
count    578.000000
mean       5.524315
std        2.195738
min        1.303838
25%        4.174432
50%        5.307981
75%        6.417345
max       13.349131
Name: recent_volatility, dtype: float64


In [6]:
# 4 week momentum — short term trend
df['momentum_4w'] = df['carbon_price'].pct_change(periods=4) * 100

# 12 week momentum — medium term trend
df['momentum_12w'] = df['carbon_price'].pct_change(periods=12) * 100

print("Momentum features added")
df[['carbon_price', 'momentum_4w', 'momentum_12w']].dropna().head(10)

Momentum features added


,carbon_price,momentum_4w,momentum_12w
date,,,
2015-04-05,6.94,7.430341,-2.664797
2015-04-12,6.84,-2.840909,0.588235
2015-04-19,7.28,7.692308,2.824859
2015-04-26,7.48,4.615385,7.625899
2015-05-03,7.53,8.501441,-1.310616
2015-05-10,7.59,10.964912,3.688525
2015-05-17,7.30,0.274725,2.816901
2015-05-24,7.32,-2.139037,7.647059
2015-05-31,7.40,-1.726428,14.551084


In [7]:
df['gas_price_volatility'] = df['gas_price_weekly']\
                               .pct_change().rolling(window=4).std() * 100

print("Gas price volatility added")
print(df['gas_price_volatility'].describe())

Gas price volatility added
count    585.000000
mean       9.718868
std       16.445543
min        0.284946
25%        3.436706
50%        5.930223
75%        9.709006
max      167.437059
Name: gas_price_volatility, dtype: float64


In [8]:
df['cop_urgency'] = np.where(
    df['days_to_next_cop'] <= 60,
    (60 - df['days_to_next_cop']) / 60,  # Score from 0 to 1 as COP approaches
    0                                      # Zero urgency beyond 60 days
)

print("COP urgency distribution:")
print(df['cop_urgency'].describe())
print("\nWeeks with non-zero urgency:", (df['cop_urgency'] > 0).sum())

COP urgency distribution:
count    589.000000
mean       0.077957
std        0.219178
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: cop_urgency, dtype: float64

Weeks with non-zero urgency: 88


In [9]:
print("Shape before dropping nulls:", df.shape)
df = df.dropna()
print("Shape after dropping nulls :", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum())

Shape before dropping nulls: (589, 15)
Shape after dropping nulls : (577, 15)

Remaining nulls:
carbon_price               0
carbon_price_change_pct    0
gas_price_weekly           0
gas_price_lag1             0
vix                        0
co2_ppm                    0
co2_change                 0
days_to_next_cop           0
is_cop_week                0
regime                     0
recent_volatility          0
momentum_4w                0
momentum_12w               0
gas_price_volatility       0
cop_urgency                0
dtype: int64


In [10]:
TARGET = 'carbon_price_change_pct'

FEATURES = [
    'gas_price_weekly',      # Energy market — strongest driver
    'gas_price_lag1',        # Lagged energy effect
    'gas_price_volatility',  # Energy shock effect
    'vix',                   # Market fear and risk sentiment
    'co2_change',            # Weekly atmospheric CO2 change
    'days_to_next_cop',      # Raw COP countdown
    'cop_urgency',           # Nonlinear COP urgency score
    'is_cop_week',           # COP summit week binary flag
    'recent_volatility',     # Market volatility memory
    'momentum_4w',           # Short term price momentum
    'momentum_12w',          # Medium term price momentum
    'regime'                 # Market phase / structural regime
]

X = df[FEATURES]
y = df[TARGET]

print(f"Features : {len(FEATURES)}")
print(f"Rows     : {len(X)}")
print(f"\nFeature list:\n{FEATURES}")

Features : 12
Rows     : 577

Feature list:
['gas_price_weekly', 'gas_price_lag1', 'gas_price_volatility', 'vix', 'co2_change', 'days_to_next_cop', 'cop_urgency', 'is_cop_week', 'recent_volatility', 'momentum_4w', 'momentum_12w', 'regime']


In [11]:
df.to_csv(
    r'C:\Users\boser\Desktop\Project_Carbon\data\processed\final_dataset_engineered.csv'
)
print("Saved final_dataset_engineered.csv")
print("Shape:", df.shape)
df.describe()

Saved final_dataset_engineered.csv
Shape: (577, 15)


,carbon_price,carbon_price_change_pct,gas_price_weekly,gas_price_lag1,vix,co2_ppm,co2_change,days_to_next_cop,is_cop_week,regime,recent_volatility,momentum_4w,momentum_12w,gas_price_volatility,cop_urgency
count,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000
mean,42.618579,0.569039,3.176291,3.176205,18.415300,414.828267,0.012786,214.331023,0.019064,1.230503,5.523848,2.289099,6.644539,9.779570,0.079578
std,30.829982,5.875052,1.530170,1.530157,6.936630,8.430689,0.164675,144.669653,0.136869,0.823213,2.197615,11.240566,19.106842,16.550203,0.221157
min,4.080000,-26.140231,1.400000,1.400000,9.340000,397.410000,-0.537773,0.000000,0.000000,0.000000,1.303838,-35.305417,-41.549296,0.284946,0.000000
25%,8.630000,-3.019608,2.380000,2.380000,13.626000,408.070000,-0.090217,99.000000,0.000000,1.000000,4.173644,-4.220913,-5.714286,3.432642,0.000000
50%,29.360000,0.588091,2.820000,2.820000,16.836000,414.540000,0.004781,200.000000,0.000000,1.000000,5.306965,1.951780,5.191595,5.934670,0.000000
75%,72.370000,3.927492,3.250000,3.250000,21.362000,421.280000,0.103316,309.000000,0.000000,2.000000,6.417696,8.614232,17.535169,9.813028,0.000000
max,98.010000,26.166329,13.800000,13.800000,74.618001,432.440000,0.840674,694.000000,1.000000,2.000000,13.349131,52.093023,71.428571,167.437059,1.000000
